[Back to Computer Organization and Architecture guideline](Computer-Organization.html)

## **Performance, Power, Reliability, and Security** {#performance-power-reliability-and-security}

The preceding chapters built a computer from abstractions, logic, instructions, pipelines, memory, I/O, and parallel execution. A real design is not finished when it produces the correct answer. It must produce that answer quickly enough, within an energy and thermal budget, despite faults, and without allowing one protection domain to infer or modify another domain's data.

These qualities are coupled. A larger cache may reduce memory traffic but consume more area and leakage power. Speculation may improve average performance while increasing state that must be isolated. Redundancy may raise availability while adding cost, energy, and common-mode failure risks. The purpose of this chapter is therefore not to maximize one number; it is to make a defensible system-level decision under explicit constraints.

A useful order of reasoning is:

1. define the workload and the user-visible requirement;
2. measure the complete system under controlled conditions;
3. locate the limiting resource instead of guessing;
4. use a quantitative model to predict what a change can improve;
5. check power, reliability, and security consequences;
6. validate the final design with the original workload.

| Design question | Representative evidence | Common mistake |
|---|---|---|
| Is it fast enough? | latency distribution, throughput, deadline misses | reporting only average CPU utilization |
| Is it energy-efficient? | joules per completed task, idle and peak power | confusing lower power with lower energy |
| Is it dependable? | error coverage, reliability, recovery time, availability | assuming redundancy removes every failure |
| Is it protected? | privilege checks, isolation boundaries, leakage analysis | checking architectural state but ignoring shared microarchitectural state |
| Is it affordable? | acquisition, cooling, operating, and engineering cost | optimizing hardware price while ignoring total lifecycle cost |

### **Evaluating a Complete Computer System** {#evaluating-a-complete-computer-system}

A **computer-system evaluation** measures whether a specified hardware and software configuration satisfies a specified workload's requirements. The wording is deliberately precise: performance is not an intrinsic property of a processor model alone. It depends on input data, compiler flags, libraries, operating-system policy, memory capacity, storage, network behavior, thermal state, and the metric chosen by the observer.

The first decision is the **system boundary**. CPU time excludes periods when a process is descheduled or waiting for I/O, whereas elapsed or wall-clock time includes everything the user waits for. CPU time is useful for diagnosing processor work; elapsed time is normally the final latency metric. For a server, one must also distinguish a single request's latency from aggregate throughput and from tail latency such as the 95th or 99th percentile.

::: {.diagram-scroll .wide-diagram}
![A complete evaluation follows the workload through software, processor, memory, and I/O, then observes user-visible and operational outcomes.](assets/system-evaluation-stack.svg)
:::

The main metrics answer different questions:

- **latency** is the time from a request or job beginning until its required result is available;
- **throughput** is useful work completed per unit time, such as requests/s, transactions/s, or bytes/s;
- **tail latency** reveals rare but operationally important slow requests that an average hides;
- **utilization** is the fraction of a resource's capacity that is occupied, but high utilization does not prove useful progress;
- **energy per task** combines power with execution duration and permits fair comparison of differently paced systems;
- **availability** measures the fraction of time the service is usable, including repair and recovery;
- **correctness and security constraints** are gates rather than optional score bonuses: a fast incorrect or leaking result is not a successful result.

A workload generator must preserve the intended arrival process. In a **closed-loop** test, the client waits for one response before issuing another request; a slow system therefore receives less offered load and can appear healthier than it is. An **open-loop** test schedules arrivals independently of completions and is better for observing queue growth and tail latency near saturation. Whichever method is used, the report must state it.

Evaluation should end with a requirement-based sentence, not merely a score: for example, "At 2,000 requests/s, 99% of requests finish within 40 ms while package energy remains below 8 J per 1,000 requests." That statement identifies the workload, operating point, percentile, and resource constraint.

### **Benchmarks and Workload Selection** {#benchmarks-and-workload-selection}

A **benchmark** is a controlled workload used to compare implementations or track changes. A benchmark is valuable only when its behavior represents the real question. Sorting one tiny array may reveal instruction overhead but says little about an analytics service whose working set exceeds memory. Conversely, a complete production trace can be realistic but difficult to reproduce or diagnose.

| Benchmark level | What it isolates | Main strength | Main limitation |
|---|---|---|---|
| microbenchmark | one operation or subsystem | explains latency, bandwidth, or instruction cost | easily optimized into an unrealistic special case |
| kernel benchmark | a recurring computation such as matrix multiply | preserves an important access or compute pattern | omits surrounding software and I/O |
| application benchmark | an end-to-end program and input | captures interactions among components | harder to attribute a change |
| benchmark suite | several programs or workload classes | reduces dependence on one program | aggregate score can hide regressions |
| production replay | recorded request or data distribution | high external validity | privacy, drift, and reproducibility challenges |

A fair experiment holds non-target variables constant: source and binary versions, compiler options, core affinity, thread count, memory placement, firmware settings, power policy, input data, cache state, and background activity. Warm-up runs allow code pages, JIT compilation, caches, and thermal control to approach the state being studied. Repeated measured runs expose noise and drift.

::: {.diagram-scroll .wide-diagram}
![A reproducible benchmark controls the workload and environment, reaches the intended state, repeats measurements, and reports the evidence rather than one convenient number.](assets/benchmark-experiment-design.svg)
:::

The [SPEC CPU run rules](https://www.spec.org/cpu2026/docs/runrules.html) are a useful model of experimental discipline: results should be meaningful, comparable, reproducible, and accompanied by enough configuration disclosure to interpret them. Their rules also separate allowed general optimization from benchmark-specific shortcuts. The lesson applies beyond SPEC: the benchmark should exercise techniques that transfer to real programs.

For repeated latency runs, report a distribution or several percentiles rather than only a minimum. When combining **normalized ratios** across different benchmarks, the geometric mean is usually appropriate:

$$
G = \left(\prod_{i=1}^{n} r_i\right)^{1/n}.
$$

Here, $r_i$ is a dimensionless performance ratio for benchmark $i$, $n$ is the number of benchmarks, and $G$ is the multiplicative central tendency. Ratios must use one consistent direction, such as "new throughput / baseline throughput." Mixing speedups with runtime ratios reverses the meaning.

<details>
<summary>Python model: summarize repeated runs and combine normalized ratios</summary>

```python
from math import prod
from statistics import median


def percentile(samples, fraction):
    """Return a linearly interpolated percentile for 0 <= fraction <= 1."""
    ordered = sorted(samples)
    position = (len(ordered) - 1) * fraction
    lower = int(position)
    upper = min(lower + 1, len(ordered) - 1)
    weight = position - lower
    return ordered[lower] * (1 - weight) + ordered[upper] * weight


def summarize_runs(runs_ms):
    # Preserve run boundaries so drift between runs is visible.
    run_medians = [median(run) for run in runs_ms]
    all_requests = [latency for run in runs_ms for latency in run]
    return {
        "run_medians_ms": run_medians,
        "median_of_run_medians_ms": median(run_medians),
        "p95_request_ms": percentile(all_requests, 0.95),
        "range_of_run_medians_ms": max(run_medians) - min(run_medians),
    }


runs = [
    [10.1, 10.4, 10.2, 11.0, 10.3],
    [10.3, 10.2, 10.5, 10.8, 10.4],
    [10.2, 10.4, 10.3, 10.9, 10.5],
]
summary = summarize_runs(runs)
print(summary)

# Ratios are all "candidate throughput / baseline throughput".
ratios = [1.18, 0.96, 1.31]
geometric_mean = prod(ratios) ** (1 / len(ratios))
print(f"geometric-mean throughput ratio = {geometric_mean:.3f}x")

assert summary["median_of_run_medians_ms"] == 10.4
assert 1.13 < geometric_mean < 1.15
```

</details>

This code does not make three short runs statistically conclusive. It demonstrates the reporting structure: retain raw samples, examine between-run drift, report tails, and aggregate ratios only after their direction and workload relevance are established.

### **Locating Performance Bottlenecks** {#locating-performance-bottlenecks}

A **bottleneck** is the resource or dependency that currently limits the metric of interest. "Currently" matters: after one limit is removed, the bottleneck can move. A faster CPU may expose a memory-bandwidth ceiling; a larger cache may move waiting time to storage; batching I/O may increase queueing latency even while throughput rises.

Bottleneck analysis begins with the application symptom and narrows progressively. First confirm that the result is correct and the slowdown is reproducible. Next inspect coarse resource activity, then use profiles, hardware performance counters, queue measurements, and traces to test a specific explanation. Counter names are architecture-specific, and some events overlap, so a counter should support a causal model rather than replace one.

::: {.diagram-scroll .wide-diagram}
![Bottleneck localization moves from the user-visible symptom through increasingly focused evidence, then validates one predicted change.](assets/bottleneck-localization.svg)
:::

A useful experiment changes one relevant lever and predicts the direction and approximate magnitude before running it. If doubling memory bandwidth does not help a supposedly bandwidth-bound kernel, the hypothesis is incomplete. If lowering CPU frequency barely changes I/O-heavy latency, the observation supports the view that processor execution is not on the critical path.

The [Linux perf security documentation](https://kernel.org/doc/html/next/admin-guide/perf-security.html) is also a reminder that profiling data can reveal addresses, execution context, register content, or workload behavior. Production measurement therefore needs access control and data minimization, not unrestricted collection simply because counters are "only performance data."

#### **Processor Bottlenecks** {#processor-bottlenecks}

A workload is **processor-bound** when useful progress is limited primarily by instruction execution, front-end supply, branch recovery, or execution-unit capacity rather than by long waits for memory or I/O. Near-100% CPU utilization is not sufficient evidence: a core may be busy spinning on a lock, repeatedly missing in cache, or executing work that should have been eliminated.

Instruction count, cycles, IPC, and CPI provide the first decomposition:

$$
\mathrm{IPC} = \frac{\text{retired instructions}}{\text{cycles}},
\qquad
\mathrm{CPI} = \frac{\text{cycles}}{\text{retired instructions}} = \frac{1}{\mathrm{IPC}}
$$

for a scalar definition over the same measurement interval. Wide superscalar processors can retire more than one instruction per cycle, so IPC can exceed one. A low IPC is a symptom, not a diagnosis. It may reflect a dependency chain, branch mispredictions, front-end starvation, cache misses, or contention for execution ports.

A conceptual exclusive CPI stack is

$$
\mathrm{CPI}
=
\mathrm{CPI}_{retire}
+
\mathrm{CPI}_{frontend}
+
\mathrm{CPI}_{branch}
+
\mathrm{CPI}_{memory}
+
\mathrm{CPI}_{other}.
$$

Each term attributes cycles per retired instruction to a category. Real hardware events are often sampled or overlapping, so vendor-specific top-down methods should be followed when constructing an actual stack. The value of the model is to ask where cycles go and which architectural change targets that category.

<details>
<summary>Python model: convert an exclusive cycle attribution into a CPI stack</summary>

```python
def cpi_stack(instructions, cycle_categories):
    total_cycles = sum(cycle_categories.values())
    if instructions <= 0 or total_cycles <= 0:
        raise ValueError("instructions and cycles must be positive")

    cpi_components = {
        name: cycles / instructions
        for name, cycles in cycle_categories.items()
    }
    limiting_category = max(cycle_categories, key=cycle_categories.get)
    return total_cycles / instructions, cpi_components, limiting_category


categories = {
    "useful_retirement": 320_000,
    "front_end_wait": 90_000,
    "branch_recovery": 70_000,
    "memory_wait": 410_000,
    "other_back_end": 110_000,
}
cpi, components, limit = cpi_stack(500_000, categories)

print(f"CPI = {cpi:.2f}")
for name, value in components.items():
    print(f"{name:18s}: {value:.2f} cycles/instruction")
print("largest category:", limit)

assert cpi == 2.0
assert limit == "memory_wait"
```

</details>

Although this synthetic example's largest category is memory waiting, the decomposition is performed from the processor's point of view. The next step is to determine whether the delay comes from cache capacity, poor locality, bandwidth saturation, NUMA placement, or a dependency that prevents multiple misses from overlapping.

#### **Memory Bottlenecks** {#memory-bottlenecks}

A workload is **memory-bound** when the rate or latency of data delivery limits progress. Three different mechanisms must be separated:

- **latency-bound behavior** waits for a dependent load before it can continue;
- **bandwidth-bound behavior** has enough concurrent requests to saturate the available transfer rate;
- **capacity or translation pressure** causes cache, TLB, or page faults because the active working set does not fit.

Cache-miss rate alone is incomplete. Ten independent misses may overlap through memory-level parallelism, while one pointer-chasing miss can stop a dependency chain. Similarly, high measured bandwidth may mean efficient streaming near the hardware ceiling, or it may mean that poor locality moves unnecessary bytes. Useful bandwidth is the rate of data that contributes to the result; physical bandwidth includes refetches, writebacks, coherence traffic, and prefetches.

Evidence should connect a symptom to a mechanism:

| Observation | Likely interpretation | Useful validation |
|---|---|---|
| high last-level-cache miss latency, bandwidth below peak | dependent or low-concurrency misses | increase memory-level parallelism or improve locality |
| bandwidth near a sustained ceiling | transfer-rate limit | reduce bytes moved or raise locality; more cores may not help |
| remote NUMA accesses | data and threads are placed apart | bind memory near the consuming cores |
| many TLB misses or page walks | translation working set is large | test larger pages or a smaller active footprint |
| major page faults | data is fetched from storage | separate memory capacity from DRAM performance |

A memory optimization should predict both **bytes moved** and **time saved**. Compression, tiling, structure-of-arrays layout, blocking, and cache-aware traversal can outperform a processor upgrade because they reduce movement through several levels at once.

#### **I/O Bottlenecks** {#io-bottlenecks}

A workload is **I/O-bound** when storage, network, device service, or its software path determines completion time. The waiting time observed by an application includes more than the device's advertised media latency:

$$
T_{I/O}
=
T_{queue}
+
T_{software}
+
T_{controller}
+
T_{device}
+
T_{transfer}.
$$

$T_{queue}$ is time waiting behind earlier requests; $T_{software}$ includes system calls, protocol processing, interrupts, and driver work; $T_{controller}$ is command handling; $T_{device}$ is media or endpoint service; and $T_{transfer}$ moves the payload. Near saturation, queueing can grow sharply even when the device's service time is unchanged.

Storage and network bottlenecks are workload-shaped. Small random requests emphasize per-request latency and IOPS. Large sequential requests emphasize transfer bandwidth. Synchronous access exposes latency directly, whereas asynchronous queues can overlap requests but may increase tail latency if the queue is too deep. Compression can reduce I/O bytes while increasing processor work, so end-to-end measurement determines whether the exchange is favorable.

Useful evidence includes queue depth, request size distribution, outstanding operations, completion latency, bytes/s, IOPS, retransmissions, interrupt rate, DMA throughput, and CPU time in the kernel or protocol stack. A CPU that is not fully utilized can still be the I/O bottleneck if one serial driver or network-processing thread is saturated. Conversely, low CPU use plus long device queues points toward the device or external service.

The processor, memory, and I/O labels are therefore hypotheses about the **critical path**, not permanent categories assigned to an application.

### **Quantitative Performance Models** {#quantitative-performance-models}

A quantitative model converts measurements and architectural assumptions into a prediction. Models are deliberately simpler than the machine: the CPU-time equation explains instruction execution, Amdahl's law limits partial enhancement, the Roofline model separates compute and bandwidth ceilings, and queueing models explain delay under load. Their purpose is to rule out impossible expectations and identify the variables worth measuring.

A model should state:

- the response variable, such as runtime, throughput, or energy;
- the units and meaning of every input;
- which terms are measured and which are assumed;
- the regime in which the model applies;
- a validation experiment that could falsify the prediction.

A model that predicts a 30% speedup and measures 3% is not useless. The discrepancy points to omitted overhead, a moved bottleneck, or an incorrect workload assumption. The correct response is to update the model, not to discard the inconvenient measurement.

#### **CPU-Time Equation** {#cpu-time-equation}

For processor execution, the central identity is

$$
T_{CPU}
=
IC \times CPI \times T_{cycle}
=
\frac{IC \times CPI}{f_{clock}}.
$$

$T_{CPU}$ is processor time in seconds, $IC$ is the number of retired instructions, $CPI$ is average cycles per retired instruction, $T_{cycle}$ is seconds per cycle, and $f_{clock}=1/T_{cycle}$ is cycles per second. The units cancel visibly: instructions multiplied by cycles/instruction and seconds/cycle gives seconds.

::: {.diagram-scroll .wide-diagram}
![Instruction count, CPI, and cycle time expose different software, microarchitectural, and technology levers in the CPU-time equation.](assets/cpu-time-levers.svg)
:::

The equation prevents the "higher frequency is always faster" mistake. A more complex implementation might raise frequency but also require more instructions or suffer more cache misses. An instruction-set extension might reduce instruction count while lengthening the critical path. Compiler vectorization may execute fewer instructions but increase memory bandwidth pressure. The final product, not one factor, determines CPU time.

<details>
<summary>Python model: compare two designs with the CPU-time equation</summary>

```python
def cpu_time_seconds(instruction_count, cpi, frequency_ghz):
    frequency_hz = frequency_ghz * 1e9
    return instruction_count * cpi / frequency_hz


baseline = cpu_time_seconds(
    instruction_count=1.20e9,
    cpi=1.80,
    frequency_ghz=3.0,
)
candidate = cpu_time_seconds(
    instruction_count=1.05e9,  # better code generation
    cpi=2.00,                  # slightly more expensive instructions
    frequency_ghz=3.4,
)
speedup = baseline / candidate

print(f"baseline CPU time = {baseline:.3f} s")
print(f"candidate CPU time = {candidate:.3f} s")
print(f"speedup = {speedup:.3f}x")

assert abs(baseline - 0.72) < 1e-12
assert 1.16 < speedup < 1.17
```

</details>

The candidate wins despite a worse CPI because instruction count and frequency compensate. This is a CPU-time comparison only; an end-to-end decision must still include memory, I/O, power, and thermal behavior.

#### **Amdahl's Law Revisited** {#amdahls-law-revisited}

Amdahl's law limits any optimization that affects only part of execution. If fraction $F$ of baseline time can use an enhancement with local speedup $S_e$, the ideal overall speedup is

$$
S_{overall}
=
\frac{1}{(1-F)+F/S_e}.
$$

The unaffected fraction $1-F$ remains even if the enhancement becomes infinitely fast. Real accelerators also require launch, transfer, conversion, synchronization, and fallback work. If normalized overhead $O$ is measured relative to baseline runtime, a more honest estimate is

$$
S_{overall}
=
\frac{1}{(1-F)+F/S_e+O}.
$$

::: {.diagram-scroll .wide-diagram}
![Acceleration shrinks only the enhanceable fraction; unaffected work and transfer or launch overhead remain in total time.](assets/amdahl-acceleration-overhead.svg)
:::

For a GPU or dedicated accelerator, $F$ must represent the fraction that can actually be offloaded at the required precision and batch size, not the fraction that merely looks arithmetically similar. $O$ must include data movement across the real interconnect and any loss of overlap. For multicore execution, synchronization and load imbalance play the same role.

<details>
<summary>Python model: include accelerator overhead in Amdahl's law</summary>

```python
def amdahl_speedup(enhanceable_fraction, local_speedup, overhead=0.0):
    if not 0 <= enhanceable_fraction <= 1:
        raise ValueError("fraction must be between 0 and 1")
    if local_speedup <= 0 or overhead < 0:
        raise ValueError("speedup must be positive and overhead non-negative")
    new_time = (
        (1 - enhanceable_fraction)
        + enhanceable_fraction / local_speedup
        + overhead
    )
    return 1 / new_time


for fraction in (0.40, 0.70, 0.95):
    ideal = amdahl_speedup(fraction, local_speedup=20)
    realistic = amdahl_speedup(fraction, local_speedup=20, overhead=0.04)
    print(
        f"F={fraction:.2f}: ideal={ideal:.2f}x, "
        f"with overhead={realistic:.2f}x"
    )

# Even infinite acceleration cannot remove the unaffected 30%.
ceiling_without_overhead = 1 / (1 - 0.70)
assert abs(ceiling_without_overhead - 3.3333333333) < 1e-9
assert amdahl_speedup(0.70, 20, 0.04) < ceiling_without_overhead
```

</details>

The strongest optimization target is not necessarily the component with the largest local speedup. It is the component that occupies a large measured fraction of end-to-end time and can be improved without excessive overhead.

#### **Bandwidth and Operational Intensity** {#bandwidth-and-operational-intensity}

The **Roofline model** estimates attainable compute performance from two ceilings: peak arithmetic throughput and sustained memory bandwidth. **Operational intensity** $I$ is the number of useful operations performed per byte transferred from the memory level being modeled:

$$
I = \frac{\text{operations}}{\text{bytes transferred}},
\qquad
P_{attainable}
=
\min(P_{peak}, I \times B_{memory}).
$$

$P_{attainable}$ and $P_{peak}$ use operations/s, $B_{memory}$ uses bytes/s, and $I$ uses operations/byte. Therefore $I B_{memory}$ has units of operations/s. The **ridge point** is $I^*=P_{peak}/B_{memory}$. Kernels left of this point are bounded by bandwidth in the simple model; kernels right of it are bounded by compute throughput.

![A naive Roofline plot places a memory-bound kernel under the sloped bandwidth ceiling and a compute-bound kernel under the horizontal peak-performance ceiling.](assets/source-roofline-model.svg)

*Image source: [Giu.natale, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Example_of_a_naive_Roofline_model.svg), licensed under [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). The conceptual model was introduced in the [original Roofline paper](https://escholarship.org/uc/item/5tz795vq).*

A memory-bound kernel can move right by reusing each byte for more operations, blocking data into cache, fusing loops, or eliminating unnecessary transfers. Raising arithmetic peak does little until the kernel crosses the ridge point. A compute-bound kernel instead benefits from vectorization, additional arithmetic units, better instruction scheduling, or a suitable accelerator. Real Roofline analyses often add ceilings for cache bandwidth, instruction mix, or limited vectorization.

<details>
<summary>Python model: classify kernels with a Roofline bound</summary>

```python
def roofline_bound(peak_ops_per_s, bandwidth_bytes_per_s, intensity):
    bandwidth_ceiling = bandwidth_bytes_per_s * intensity
    attainable = min(peak_ops_per_s, bandwidth_ceiling)
    limiting_resource = (
        "memory bandwidth"
        if bandwidth_ceiling < peak_ops_per_s
        else "compute throughput"
    )
    return attainable, limiting_resource


peak = 2.0e12       # 2 TOP/s
bandwidth = 100e9   # 100 GB/s
ridge_point = peak / bandwidth
print(f"ridge point = {ridge_point:.1f} operations/byte")

kernels = {
    "streaming_filter": 2.0,
    "blocked_matrix_kernel": 35.0,
}
for name, intensity in kernels.items():
    attainable, limit = roofline_bound(peak, bandwidth, intensity)
    print(f"{name}: {attainable / 1e9:.0f} GOP/s, limited by {limit}")

assert ridge_point == 20.0
assert roofline_bound(peak, bandwidth, 2.0)[0] == 200e9
assert roofline_bound(peak, bandwidth, 35.0)[0] == peak
```

</details>

Operational intensity must be measured at a named boundary. Bytes transferred between registers and L1, between the last-level cache and DRAM, and between host and accelerator produce different Rooflines and different optimization decisions.

### **Power and Energy** {#power-and-energy}

**Power** is the rate of energy use, measured in watts ($1\ \mathrm{W}=1\ \mathrm{J/s}$). **Energy** is the accumulated work-related consumption, measured in joules. For time-varying power,

$$
E = \int_0^T P(t)\,dt
\approx
\sum_i P_i \Delta t_i,
\qquad
P_{avg} = \frac{E}{T}.
$$

A low-power design can consume more energy if it runs long enough; a high-power burst can use less total energy if it finishes quickly and returns the system to an efficient idle state. Consequently, mobile devices care about battery energy, data centers care about power delivery and cooling as well as energy cost, and real-time systems care about whether a power-limited configuration still meets its deadline.

Measurement must match the boundary. On-chip telemetry can estimate package or domain energy with low overhead, while a calibrated external meter captures memory, power-supply losses, storage, fans, and other components. The Linux [powercap framework](https://docs.kernel.org/6.8/power/powercap/powercap.html) exposes energy counters and power constraints for supported zones; accumulated energy over a measured interval is generally more meaningful than treating such telemetry as an instantaneous wall-power reading.

#### **Dynamic and Static Power** {#dynamic-and-static-power}

In CMOS logic, a useful first-order decomposition is

$$
P_{total}
\approx
P_{dynamic}+P_{static},
$$

with

$$
P_{dynamic}
\approx
\alpha C_{eff} V^2 f,
\qquad
P_{static}
\approx
I_{leak}V.
$$

$\alpha$ is the activity factor, the fraction of capacitance switching per cycle; $C_{eff}$ is effective switched capacitance; $V$ is supply voltage; $f$ is clock frequency; and $I_{leak}$ is leakage current. The dynamic term's $V^2$ dependence is crucial: frequency increases often require a voltage increase, so power can rise faster than frequency.

::: {.diagram-scroll .wide-diagram}
![Dynamic switching power and static leakage form total power; integrating that power over runtime gives task energy.](assets/power-energy-model.svg)
:::

Clock gating lowers activity by stopping clocks to idle structures. Power gating reduces leakage by disconnecting a region, but waking it costs time and energy and may require state retention. Smaller or simpler structures can reduce capacitance and leakage, while aggressive speculation increases switched work that may never retire. Therefore "performance per watt" depends on useful completed work, not raw switching activity.

<details>
<summary>Python model: inspect voltage, frequency, and leakage sensitivity</summary>

```python
def cmos_power(activity, capacitance_f, voltage, frequency_hz, leakage_a):
    dynamic = activity * capacitance_f * voltage**2 * frequency_hz
    static = leakage_a * voltage
    return {
        "dynamic_w": dynamic,
        "static_w": static,
        "total_w": dynamic + static,
    }


low = cmos_power(
    activity=0.20,
    capacitance_f=40e-9,
    voltage=0.80,
    frequency_hz=2.0e9,
    leakage_a=4.0,
)
high = cmos_power(
    activity=0.20,
    capacitance_f=40e-9,
    voltage=1.00,
    frequency_hz=3.0e9,
    leakage_a=5.0,
)

for label, point in (("low", low), ("high", high)):
    print(label, {name: round(value, 2) for name, value in point.items()})

power_ratio = high["total_w"] / low["total_w"]
frequency_ratio = 3.0 / 2.0
print(f"power ratio={power_ratio:.2f}, frequency ratio={frequency_ratio:.2f}")

assert power_ratio > frequency_ratio
```

</details>

This is a sensitivity model, not a transistor-level estimator. Real processors contain multiple voltage domains, regulators, clock trees, memories, analog components, and temperature-dependent leakage.

#### **Voltage and Frequency Scaling** {#voltage-and-frequency-scaling}

**Dynamic voltage and frequency scaling (DVFS)** selects among supported operating points. Lowering frequency reduces switching rate; lowering voltage provides the larger quadratic dynamic-power benefit, but voltage cannot normally be reduced arbitrarily while preserving timing. A system may expose discrete performance states and use governors or firmware to choose among them according to load, latency targets, temperature, and power caps.

The [Linux CPUFreq documentation](https://docs.kernel.org/6.7/admin-guide/pm/cpufreq.html) describes this policy/mechanism split: scaling drivers know the hardware performance states, while governors select a target within policy limits. Modern hardware may also make fast autonomous decisions below the operating system.

::: {.diagram-scroll .wide-diagram}
![DVFS operating points trade sharply rising power for shorter compute time, while memory-stall time may remain almost unchanged.](assets/dvfs-operating-points.svg)
:::

The energy-optimal point depends on workload composition. For compute-bound work, a higher frequency can shorten runtime substantially. For memory-bound work, raising only the core clock may increase power while dependent DRAM time barely changes. "Race to idle" is beneficial only when the higher-power state finishes soon enough and the resulting idle state is efficient.

<details>
<summary>Python model: choose the lowest-energy DVFS point that meets a deadline</summary>

```python
def dvfs_result(voltage, frequency_ghz, static_power_w,
                compute_cycles, memory_time_s,
                activity=0.20, capacitance_f=40e-9):
    frequency_hz = frequency_ghz * 1e9
    compute_time = compute_cycles / frequency_hz
    runtime = compute_time + memory_time_s
    dynamic_power = (
        activity * capacitance_f * voltage**2 * frequency_hz
    )
    total_power = dynamic_power + static_power_w
    return {
        "runtime_s": runtime,
        "power_w": total_power,
        "energy_j": total_power * runtime,
    }


states = {
    "eco": (0.75, 1.5, 4.0),
    "balanced": (0.90, 2.5, 5.0),
    "turbo": (1.05, 3.2, 7.0),
}
results = {
    name: dvfs_result(v, f, static, 4e9, memory_time_s=0.60)
    for name, (v, f, static) in states.items()
}

deadline_s = 2.5
feasible = {
    name: result
    for name, result in results.items()
    if result["runtime_s"] <= deadline_s
}
choice = min(feasible, key=lambda name: feasible[name]["energy_j"])

for name, result in results.items():
    print(name, {key: round(value, 2) for key, value in result.items()})
print("lowest-energy state meeting deadline:", choice)

assert choice == "balanced"
```

</details>

The result is deliberately requirement-driven: eco uses the least energy in this toy model but misses the deadline, while balanced meets it with less energy than turbo.

#### **Thermal Constraints** {#thermal-constraints}

Electrical power becomes heat, and temperature evolves more slowly than instruction execution. A simple lumped thermal model is

$$
C_{th}\frac{dT}{dt}
=
P(t)-\frac{T(t)-T_{ambient}}{R_{th}}.
$$

$T(t)$ is device temperature, $T_{ambient}$ is inlet or surrounding temperature, $P(t)$ is heat-producing power, $R_{th}$ is thermal resistance in degrees Celsius per watt, and $C_{th}$ is thermal capacitance in joules per degree Celsius. The product $\tau=R_{th}C_{th}$ is a thermal time constant: it describes how quickly temperature approaches a new equilibrium.

::: {.diagram-scroll .wide-diagram}
![Power raises temperature, temperature increases leakage, cooling removes heat, and a control loop may reduce voltage or frequency to preserve the thermal limit.](assets/thermal-feedback-loop.svg)
:::

A processor can benchmark quickly in a temporary boost state but throttle during a long workload. Short, cold runs therefore overestimate sustained performance. Cooling design, ambient temperature, fan policy, package contact, and neighboring devices are part of the performance experiment. Intel's [system-overview analysis guidance](https://www.intel.com/content/www/us/en/docs/vtune-profiler/user-guide/2025-4/system-overview-analysis.html) explicitly connects frequency, power, bandwidth, and throttling observations.

<details>
<summary>Python model: simulate a first-order thermal control loop</summary>

```python
def simulate_temperature(
    seconds=120,
    dt=0.2,
    ambient_c=25.0,
    thermal_resistance_c_per_w=1.8,
    thermal_capacitance_j_per_c=25.0,
    throttle_threshold_c=78.0,
):
    temperature = ambient_c
    samples = []

    for step in range(int(seconds / dt)):
        # The controller lowers package power above the threshold.
        power_w = 38.0 if temperature < throttle_threshold_c else 20.0

        heat_in = power_w
        heat_out = (
            (temperature - ambient_c) / thermal_resistance_c_per_w
        )
        d_temperature = (
            (heat_in - heat_out)
            / thermal_capacitance_j_per_c
            * dt
        )
        temperature += d_temperature
        samples.append((step * dt, temperature, power_w))

    return samples


trace = simulate_temperature()
peak_temperature = max(sample[1] for sample in trace)
throttled_samples = sum(power == 20.0 for _, _, power in trace)

print(f"final temperature = {trace[-1][1]:.1f} C")
print(f"peak temperature = {peak_temperature:.1f} C")
print(f"throttled intervals = {throttled_samples}")

assert throttled_samples > 0
assert peak_temperature < 79.0
```

</details>

This model illustrates feedback rather than predicting a specific package. Real systems have several sensors, spatial hot spots, delayed measurements, fans, heat spreaders, and control policies with hysteresis.

### **Reliability and Fault Tolerance** {#reliability-and-fault-tolerance}

A **fault** is an underlying defect or disturbance, such as a flipped storage bit or failed link. An **error** is an incorrect internal state caused by a fault. A **failure** occurs when the system's externally required service deviates from its specification. Fault tolerance prevents some faults from becoming failures through detection, correction, isolation, redundancy, and recovery.

Faults may be **transient** (a one-time disturbance), **intermittent** (dependent on conditions), or **permanent** (a lasting defect). Radiation-induced single-event upsets, voltage noise, wear, manufacturing defects, overheating, firmware mistakes, and link failures demand different mechanisms. NASA's [small-spacecraft avionics guidance](https://www.nasa.gov/smallsat-institute/sst-soa/small-spacecraft-avionics/) discusses radiation effects together with ECC, scrubbing, watchdogs, and redundancy, illustrating why fault assumptions must match the environment.

For a constant failure rate $\lambda$, an exponential reliability model is

$$
R(t)=e^{-\lambda t},
\qquad
MTTF=\frac{1}{\lambda}.
$$

$R(t)$ is the probability of surviving without failure through time $t$, and MTTF is mean time to failure. Repairable systems additionally use

$$
A=\frac{MTBF}{MTBF+MTTR},
$$

where availability $A$ depends on mean time between failures and mean time to repair. High reliability reduces failure frequency; fast recovery reduces outage duration. A system can have imperfect component reliability yet high availability if faults are detected and service is restored quickly.

The [NIST fault-tolerant definition](https://csrc.nist.gov/glossary/term/fault_tolerant) emphasizes continued correct operation after hardware or software faults. That promise must name the covered fault model: one failed disk, one corrupted bit, one unavailable replica, or a regional outage are not interchangeable claims.

#### **Parity and Error-Correcting Codes** {#parity-and-error-correcting-codes}

**Parity** appends one bit so the total number of ones is even or odd. It detects any odd number of bit flips but cannot identify which bit changed and misses every even-numbered pattern. An **error-correcting code (ECC)** adds enough structured redundancy to locate and correct selected errors.

The key property is **minimum Hamming distance** $d_{min}$, the smallest number of bit positions in which any two valid codewords differ. A code can detect up to $d_{min}-1$ bit errors and correct up to

$$
\left\lfloor \frac{d_{min}-1}{2} \right\rfloor
$$

errors, because the received word must remain closer to the original codeword than to any other valid one.

For a Hamming code with $m$ data bits and $r$ parity bits, single-error correction requires

$$
2^r \ge m+r+1.
$$

The $2^r$ parity syndromes must distinguish every one of the $m+r$ bit positions plus the no-error case. In Hamming(7,4), parity bits occupy positions 1, 2, and 4; four data bits occupy positions 3, 5, 6, and 7. Each parity bit covers positions whose binary index contains its corresponding bit.

![Hamming(7,4) uses three overlapping parity sets to encode four data bits and identify one erroneous position.](assets/source-hamming-7-4.svg)

*Image source: [Cburnett, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Hamming(7,4).svg), available under [CC BY-SA 3.0](https://creativecommons.org/licenses/by-sa/3.0/).*

A Hamming code has distance three and provides single-error correction. Adding an overall parity bit creates common **SECDED** protection: single-error correction and double-error detection. Memory controllers also scrub memory by reading, correcting, and rewriting data before multiple latent errors accumulate in one codeword.

<details>
<summary>Python model: encode, diagnose, and correct a Hamming(7,4) word</summary>

```python
PARITY_POSITIONS = (1, 2, 4)
DATA_POSITIONS = (3, 5, 6, 7)


def hamming74_encode(data_bits):
    if len(data_bits) != 4 or any(bit not in (0, 1) for bit in data_bits):
        raise ValueError("data_bits must contain exactly four binary values")

    code = [0] * 8  # index 0 is unused; positions are 1 through 7
    for position, bit in zip(DATA_POSITIONS, data_bits):
        code[position] = bit

    # Even parity: each parity bit makes its covered XOR equal zero.
    for parity_position in PARITY_POSITIONS:
        parity = 0
        for position in range(1, 8):
            if position & parity_position:
                parity ^= code[position]
        code[parity_position] = parity

    return code[1:]


def hamming74_decode(received_bits):
    code = [0] + list(received_bits)
    syndrome = 0

    for parity_position in PARITY_POSITIONS:
        parity = 0
        for position in range(1, 8):
            if position & parity_position:
                parity ^= code[position]
        if parity:
            syndrome += parity_position

    if syndrome:
        code[syndrome] ^= 1

    data = [code[position] for position in DATA_POSITIONS]
    return data, syndrome, code[1:]


data = [1, 0, 1, 1]
encoded = hamming74_encode(data)
corrupted = encoded.copy()
corrupted[5] ^= 1  # zero-based index 5 is Hamming position 6

decoded, corrected_position, corrected_code = hamming74_decode(corrupted)
print("encoded:", encoded)
print("received:", corrupted)
print("corrected position:", corrected_position)
print("decoded:", decoded)

assert corrected_position == 6
assert decoded == data
assert corrected_code == encoded
```

</details>

The decoder assumes at most one bit error. Without the extra SECDED parity bit, some multi-bit patterns can produce a misleading syndrome and be miscorrected.

#### **Redundancy and Recovery** {#redundancy-and-recovery}

Redundancy can be placed in information, space, or time:

- **information redundancy** adds check bits, checksums, or coded fragments;
- **spatial redundancy** duplicates components or replicas so another can continue;
- **temporal redundancy** retries an operation or restores an earlier checkpoint;
- **diversity** uses different implementations or failure domains to reduce common-mode faults.

**Triple modular redundancy (TMR)** runs three modules and applies majority voting. If each module is independently correct with probability $R$ and the voter is perfect, the system is correct when all three work or exactly two work:

$$
R_{TMR}
=
R^3 + 3R^2(1-R)
=
3R^2-2R^3.
$$

![Triple modular redundancy sends the same input to three modules and uses a majority voter to mask one differing output.](assets/source-triple-modular-redundancy.jpg)

*Image source: [IjonTichyIjonTichy, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Triple_Modular_Redundancy.JPG), dedicated under [CC0 1.0](https://creativecommons.org/publicdomain/zero/1.0/).*

TMR helps only under its assumptions. A shared power supply, common design bug, environmental event, or faulty voter can defeat all replicas. Replicas therefore need fault-domain separation, health monitoring, and a repair or replacement path.

Recovery mechanisms trade lost work against overhead. Frequent checkpoints shorten rollback after a failure but consume bandwidth and storage during normal execution. Retries handle transient faults but can amplify overload when failures are correlated. Failover reduces interruption only if detection is accurate and the standby has sufficiently current state.

<details>
<summary>Python model: compare component reliability, TMR, and repair availability</summary>

```python
def tmr_reliability(component_reliability):
    r = component_reliability
    return 3 * r**2 - 2 * r**3


def availability(mtbf_hours, mttr_hours):
    return mtbf_hours / (mtbf_hours + mttr_hours)


component_r = 0.98
tmr_r = tmr_reliability(component_r)
service_availability = availability(
    mtbf_hours=2_000,
    mttr_hours=0.5,
)

print(f"one module reliability = {component_r:.6f}")
print(f"idealized TMR reliability = {tmr_r:.6f}")
print(f"repairable-service availability = {service_availability:.6%}")

assert tmr_r > component_r
assert service_availability > 0.999
```

</details>

The numerical gain is an upper bound because independence and a perfect voter were assumed. A reliability argument is complete only when it also identifies detection coverage, common-mode failures, state repair, and behavior after redundancy has been consumed.

### **Hardware Protection and Privilege** {#hardware-protection-and-privilege}

Hardware protection separates principals with different authority. A user program should not reconfigure address translation, access another process's pages, program arbitrary DMA, or disable interrupts. Privilege modes, page permissions, traps, physical protection, and device isolation make these rules enforceable even when application code is buggy or hostile.

A common pattern is:

- **user mode** executes applications with restricted instructions and user-accessible pages;
- **supervisor or kernel mode** manages processes, virtual memory, devices, and system calls;
- a more privileged **machine or firmware mode** initializes the platform and controls low-level resources;
- a synchronous exception or asynchronous interrupt transfers control to a checked handler, which validates the request before acting.

::: {.diagram-scroll .wide-diagram}
![Privilege state, virtual-memory permissions, and physical protection checks cooperate before an instruction or memory access is allowed.](assets/privilege-protection-path.svg)
:::

Page-table entries commonly contain user/supervisor and read/write/execute permissions. Execute-disable prevents data pages from being used as code; write protection prevents unintended modification; user bits stop applications from reaching kernel mappings. Physical memory protection adds checks over physical ranges, and an IOMMU extends translation and permissions to DMA-capable devices.

In RISC-V, [Physical Memory Protection](https://docs.riscv.org/reference/isa/priv/machine.html) entries define physical ranges and read, write, and execute permissions. Access checks compose with page-based translation rather than replacing it. A denied operation raises an access fault, preserving the architectural boundary.

<details>
<summary>Python model: compose privilege, page, and physical-range checks</summary>

```python
def authorize_access(mode, operation, page, physical_region):
    if mode not in {"U", "S", "M"}:
        return False, "unknown privilege mode"

    if operation == "machine_control":
        return (
            (True, "machine privilege")
            if mode == "M"
            else (False, "privileged instruction")
        )

    if mode == "U" and not page["user"]:
        return False, "page is supervisor-only"

    required_permission = {
        "read": "r",
        "write": "w",
        "execute": "x",
    }.get(operation)
    if required_permission is None:
        return False, "unknown operation"
    if not page[required_permission]:
        return False, "page permission denied"
    if not physical_region[required_permission]:
        return False, "physical protection denied"

    return True, "all hardware checks passed"


user_code_page = {"user": True, "r": True, "w": False, "x": True}
protected_region = {"r": True, "w": False, "x": True}

assert authorize_access(
    "U", "execute", user_code_page, protected_region
)[0]
assert not authorize_access(
    "U", "write", user_code_page, protected_region
)[0]
assert not authorize_access(
    "S", "machine_control", user_code_page, protected_region
)[0]
```

</details>

This model omits many details, but it captures the composition principle: an access succeeds only if every relevant layer permits it. Software validation is important, yet hardware checks form the final enforcement point.

### **Microarchitectural Security** {#microarchitectural-security}

The instruction-set architecture defines which values software is allowed to observe. The microarchitecture adds hidden state to execute that contract quickly: caches, TLBs, branch predictors, prefetchers, queues, speculative buffers, execution ports, frequency controllers, and performance counters. If two protection domains share this state, one domain may infer another's behavior from timing or contention even when architectural access checks are correct.

A **side channel** is an unintended observation path correlated with protected information. It differs from a direct permission failure: the attacker may never read the victim's address architecturally, but may observe that a cache line is warm, a branch predictor changed, or a shared unit became busy. A **covert channel** is similar infrastructure used deliberately by cooperating parties to communicate across a forbidden boundary.

Microarchitectural security analysis therefore needs five questions:

1. What secret or protected event exists?
2. Which microarchitectural state can it influence?
3. Which other domain shares or can probe that state?
4. What timing source or counter exposes the effect?
5. Which mechanism prevents, partitions, flushes, masks, or bounds the channel?

Mitigation may reduce performance through barriers, partitioned caches, disabled sharing, predictor flushing, less aggressive speculation, reduced timer precision, or constant-work software. The security boundary and threat model must be stated before judging that cost.

#### **Speculation and Side Channels** {#speculation-and-side-channels}

Speculative execution predicts control flow or data dependencies and performs work before all conditions are known. If the prediction is correct, latency is hidden. If it is wrong, the processor prevents speculative instructions from **retiring**, restores architectural state, and resumes the correct path. The security problem is that rollback may not erase every cache, TLB, predictor, or contention effect created by transient work.

::: {.diagram-scroll .wide-diagram}
![A speculative side channel arises when transient execution changes shared microarchitectural state that remains observable after architectural rollback.](assets/speculative-side-channel.svg)
:::

The [Spectre paper](https://arxiv.org/abs/1801.01203) showed that mistrained prediction can induce transient operations that encode protected data into measurable microarchitectural state across isolation boundaries. The [Meltdown paper](https://arxiv.org/abs/1801.01207) demonstrated transient use of data from a faulting access on affected implementations. They are related but not identical:

| Property | Spectre-style mechanism | Meltdown-style mechanism |
|---|---|---|
| transient cause | prediction follows an unsafe path | faulting or disallowed access is transiently used |
| violated assumption | software bounds/control dependency constrains execution | permission fault prevents any data-dependent effect |
| affected scope | broad class of speculative implementations and gadgets | implementation-specific permission-check timing |
| mitigation examples | speculation barriers, masking, retpolines, predictor controls | hardware changes, isolation, platform-specific controls |

No single mitigation covers every variant. Defenses combine hardware behavior, firmware, operating-system isolation, compiler transformations, and source-level bounds discipline. They should be evaluated against a named threat model and measured because barriers and isolation can reduce instruction-level parallelism or cache sharing.

The important architectural lesson is that "does not retire" is weaker than "leaves no observable effect." Security reasoning must include transient data flow and shared state, not only committed registers and memory.

#### **Timing and Cache Leakage** {#timing-and-cache-leakage}

Caches create a large timing difference between a hit and a miss. If secret data controls which address is touched, later access timing can reveal information about that choice. Similar leakage can arise from secret-dependent branches, variable-time instructions, TLB behavior, shared ports, power management, or early-exit comparisons.

::: {.diagram-scroll .wide-diagram}
![A secret-dependent memory address creates different cache states, and a later fast or slow observation can encode information.](assets/cache-timing-channel.svg)
:::

Intel's [timing side-channel guidance](https://www.intel.com/content/www/us/en/developer/articles/technical/software-security-guidance/secure-coding/mitigate-timing-side-channel-crypto-implementation.html) recommends designing cryptographic operations so timing and shared-resource use do not depend on secrets. **Constant-time** means more than adding a delay: control flow, memory addresses, instruction selection, and variable-latency operations must be reviewed for the target compiler and hardware.

Possible defenses include:

- constant-work algorithms and table-free cryptographic implementations;
- partitioning or avoiding shared caches and execution contexts;
- flushing selected state at protection-domain transitions;
- process and VM isolation, core scheduling, and disabling unnecessary sharing;
- reducing exposed timer or counter precision as defense in depth;
- adding hardware structures whose allocation and replacement do not cross the protected boundary.

Random noise raises the number of observations but does not remove an underlying statistical channel. Similarly, encrypting memory protects stored contents but does not automatically hide addresses or timing.

<details>
<summary>Python model: compare secret-indexed and fixed-footprint access patterns</summary>

```python
def secret_indexed_lookup(table, secret_index):
    # The touched location itself depends on the secret.
    touched = [secret_index]
    return table[secret_index], touched


def fixed_footprint_select(table, secret_index):
    # Visit every location and select one value without an early exit.
    # This illustrates the access pattern; Python is not a constant-time runtime.
    selected = 0
    touched = []
    for index, value in enumerate(table):
        choose = int(index == secret_index)
        selected = choose * value + (1 - choose) * selected
        touched.append(index)
    return selected, touched


table = [17, 29, 43, 61]
direct_value, direct_trace = secret_indexed_lookup(table, 2)
fixed_value, fixed_trace = fixed_footprint_select(table, 2)

print("secret-indexed touches:", direct_trace)
print("fixed-footprint touches:", fixed_trace)

assert direct_value == fixed_value == 43
assert direct_trace == [2]
assert fixed_trace == [0, 1, 2, 3]
```

</details>

The fixed-footprint version demonstrates the design goal but is not a production constant-time primitive: an interpreter, compiler, and processor may transform operations or introduce other data-dependent behavior. Security-critical code should use reviewed platform-specific libraries.

### **Performance, Cost, Power, and Security Tradeoffs** {#performance-cost-power-and-security-tradeoffs}

Architecture design is multi-objective. A candidate **dominates** another if it is no worse on every relevant objective and strictly better on at least one. Designs that are not dominated form a **Pareto frontier**. A frontier point is efficient, but not automatically appropriate: requirements choose among incomparable points.

::: {.diagram-scroll .wide-diagram}
![A Pareto frontier separates efficient architecture choices from dominated designs; service, power, cost, reliability, and security constraints select among frontier points.](assets/architecture-pareto-frontier.svg)
:::

Several compact metrics help compare a constrained set of designs:

$$
EDP=E\times T,
\qquad
ED^2P=E\times T^2.
$$

$E$ is energy per task and $T$ is task time. EDP balances energy and delay; $ED^2P$ penalizes delay more strongly. Neither metric replaces hard constraints: a design that misses a real-time deadline or violates an isolation requirement is infeasible regardless of its score.

| Architectural choice | Likely benefit | Likely cost or risk |
|---|---|---|
| deeper speculation and wider issue | higher single-thread performance | power, area, recovery complexity, larger transient surface |
| larger shared cache | fewer capacity misses, better sharing | leakage, latency, contention, side-channel surface |
| more cores | throughput and parallel capacity | coherence traffic, synchronization, software complexity |
| accelerator | high performance per watt for a narrow kernel | transfer overhead, programmability, verification, fallback path |
| ECC and redundancy | fault detection, correction, availability | capacity, latency, energy, voter and common-mode concerns |
| stronger partitioning and barriers | improved isolation | lost sharing, reduced speculation, context-switch overhead |

<details>
<summary>Python model: remove dominated architecture candidates</summary>

```python
candidates = [
    {"name": "eco",         "performance": 100, "energy": 40,  "cost": 1, "risk": 2},
    {"name": "balanced",    "performance": 180, "energy": 65,  "cost": 2, "risk": 2},
    {"name": "server",      "performance": 260, "energy": 120, "cost": 4, "risk": 1},
    {"name": "accelerated", "performance": 320, "energy": 130, "cost": 6, "risk": 2},
    {"name": "dominated",   "performance": 150, "energy": 90,  "cost": 3, "risk": 3},
]


def dominates(left, right):
    no_worse = (
        left["performance"] >= right["performance"]
        and left["energy"] <= right["energy"]
        and left["cost"] <= right["cost"]
        and left["risk"] <= right["risk"]
    )
    strictly_better = any([
        left["performance"] > right["performance"],
        left["energy"] < right["energy"],
        left["cost"] < right["cost"],
        left["risk"] < right["risk"],
    ])
    return no_worse and strictly_better


frontier = [
    candidate
    for candidate in candidates
    if not any(
        dominates(other, candidate)
        for other in candidates
        if other is not candidate
    )
]

print("Pareto frontier:", [candidate["name"] for candidate in frontier])
assert "dominated" not in {candidate["name"] for candidate in frontier}
assert "balanced" in {candidate["name"] for candidate in frontier}
```

</details>

The objectives and their units must be chosen before computing a frontier. A vague "risk" score may be useful for teaching, but a real design should replace it with explicit security properties, exposure, mitigation coverage, and acceptance criteria.

### **A Practical Architecture Evaluation Workflow** {#a-practical-architecture-evaluation-workflow}

The final workflow combines the chapter's ideas into a repeatable engineering loop.

::: {.diagram-scroll .wide-diagram}
![A practical architecture evaluation defines requirements, establishes a baseline, localizes and models a bottleneck, changes one variable, validates every constraint, and records the evidence.](assets/architecture-evaluation-workflow.svg)
:::

1. **Define the decision.** Name the workload, input scale, user-visible metric, required percentile or throughput, power and thermal envelope, reliability target, security boundary, and budget.
2. **Establish a reproducible baseline.** Record hardware, firmware, operating system, compiler, libraries, power policy, affinity, memory placement, storage state, input data, warm-up, and raw repeated results.
3. **Localize the bottleneck.** Move from end-to-end symptoms to resource utilization, profiles, counters, queues, and traces. Form one causal hypothesis.
4. **Select the smallest adequate model.** Use the CPU-time equation, CPI attribution, Amdahl's law, Roofline, thermal, reliability, or queueing model whose assumptions match the hypothesis.
5. **Predict before changing.** Estimate the expected direction and upper bound. Include transfer, synchronization, recovery, and measurement overhead.
6. **Change one variable.** Preserve controls so the result can be attributed.
7. **Validate end to end.** Recheck correctness, latency distributions, throughput, energy/task, temperature, fault coverage, and isolation. Look for a moved bottleneck and regressions in other workloads.
8. **Record the decision.** Keep raw measurements, model inputs, rejected alternatives, tradeoff rationale, and a regression threshold.

<details>
<summary>Python model: accept a candidate only when every hard constraint passes</summary>

```python
def evaluate_candidate(baseline, candidate, constraints):
    report = {
        "speedup": baseline["latency_ms"] / candidate["latency_ms"],
        "energy_change": (
            candidate["energy_j"] / baseline["energy_j"] - 1
        ),
        "cost_change": candidate["cost"] / baseline["cost"] - 1,
    }

    checks = {
        "latency": candidate["latency_ms"] <= constraints["max_latency_ms"],
        "energy": candidate["energy_j"] <= constraints["max_energy_j"],
        "temperature": candidate["temperature_c"] <= constraints["max_temperature_c"],
        "availability": candidate["availability"] >= constraints["min_availability"],
        "security": constraints["required_controls"].issubset(
            candidate["security_controls"]
        ),
    }
    report["checks"] = checks
    report["accepted"] = all(checks.values())
    return report


baseline = {
    "latency_ms": 50,
    "energy_j": 12,
    "temperature_c": 70,
    "availability": 0.9990,
    "cost": 100,
    "security_controls": {"nx", "iommu"},
}
candidate = {
    "latency_ms": 34,
    "energy_j": 10,
    "temperature_c": 76,
    "availability": 0.9998,
    "cost": 125,
    "security_controls": {"nx", "iommu", "ecc"},
}
constraints = {
    "max_latency_ms": 40,
    "max_energy_j": 11,
    "max_temperature_c": 80,
    "min_availability": 0.9995,
    "required_controls": {"nx", "iommu"},
}

decision = evaluate_candidate(baseline, candidate, constraints)
print(decision)
assert decision["accepted"]
assert decision["speedup"] > 1.4
```

</details>

This final gate prevents an attractive speedup from silently buying an unacceptable thermal, availability, or security regression. The outcome should be a traceable claim: under a named workload and configuration, the candidate satisfies every hard constraint and improves the selected objective by a measured amount.

Across the complete computer-organization series, one principle remains constant: abstractions make systems understandable, but quantitative evidence determines whether an implementation fulfills its contract.